In [ ]:
!pip install -q transformers accelerate datasets evaluate rouge-score bert-score torch torchvision torchaudio bitsandbytes

from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_Token"), add_to_git_credential=True)

import torch
import time
import gc
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
import evaluate
import re
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ============================================================
# 🧩 STEP 2: Load GSM8K dataset (100 samples)
# ============================================================
dataset = load_dataset("openai/gsm8k", "main")
test_data = dataset["test"].select(range(100))  # first 100 rows

# ============================================================
# 🧩 STEP 3: Regex for single-word / numeric extraction
# ============================================================
def extract_single_word_answer(output):
    match = re.search(r'\b\w+\b', output)
    if match:
        return match.group(0)
    return output.strip()

def extract_numeric_answer(text):
    match = re.search(r'-?\d+\.?\d*', text)
    if match:
        return match.group(0)
    return text.strip()

# ============================================================
# 🧩 STEP 4: Metrics functions
# ============================================================
from rouge_score import rouge_scorer

def calculate_accuracy_numeric(preds, refs):
    correct = 0
    for p, r in zip(preds, refs):
        try:
            gt = re.search(r'-?\d+\.?\d*', r.split("####")[-1]).group(0)
            pred_num = extract_numeric_answer(p)
            if gt == pred_num:
                correct += 1
        except:
            continue
    return correct / len(preds)

def calculate_exact_match_numeric(preds, refs):
    matches = 0
    for p, r in zip(preds, refs):
        try:
            gt = re.search(r'-?\d+\.?\d*', r.split("####")[-1]).group(0)
            pred_num = extract_numeric_answer(p)
            if gt == pred_num:
                matches += 1
        except:
            continue
    return matches / len(preds)

def compute_metrics(preds, refs):
    rouge = evaluate.load("rouge")
    bertscore = evaluate.load("bertscore")

    results = {}
    # ROUGE
    rouge_scores = rouge.compute(predictions=preds, references=refs)
    results.update({f"ROUGE_{k}": v for k, v in rouge_scores.items()})

    # BERTScore
    bert_scores = bertscore.compute(predictions=preds, references=refs, lang="en")
    results["BERTScore_F1"] = sum(bert_scores["f1"]) / len(bert_scores["f1"])

    # Numeric Exact Match
    results["Exact_Match"] = calculate_exact_match_numeric(preds, refs)

    return results

# ============================================================
# 🧩 STEP 5: Batched Inference function (fixed for Dataset)
# ============================================================
def run_inference_batched(model_name, dataset, num_samples=100, batch_size=8):
    """
    Runs inference in batches to avoid OOM on GPU.
    Handles HuggingFace Dataset object properly.
    """
    torch.cuda.empty_cache()
    gc.collect()

    print(f"\n🚀 Running inference for: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True
    )

    model.eval()

    # Slice dataset subset first
    dataset_subset = dataset.select(range(num_samples))

    prompt_prefix = "Provide ONLY the one-word numeric answer (no punctuation, no explanation).\nQuestion: "
    inputs = [f"{prompt_prefix}{item['question']}\nOne-word Answer:" for item in dataset_subset]
    refs = [item['answer'] for item in dataset_subset]
    preds = []

    start_time = time.time()

    # Process in batches
    for i in tqdm(range(0, len(inputs), batch_size), desc="Generating answers (batched)"):
        batch_inputs = inputs[i:i+batch_size]
        tokenized = tokenizer(batch_inputs, return_tensors="pt", padding=True, truncation=True).to(device)
        with torch.no_grad():
            output = model.generate(**tokenized, max_new_tokens=64)
        # Extract answers
        for out in output:
            pred = tokenizer.decode(out, skip_special_tokens=True)
            pred_clean = extract_numeric_answer(pred)
            preds.append(pred_clean)

    inference_time = time.time() - start_time

    # Compute metrics
    results = compute_metrics(preds, refs)
    results["Accuracy"] = calculate_accuracy_numeric(preds, refs)
    results["Inference_Time(s)"] = round(inference_time, 2)

    # Memory and model size info
    gpu_mem = torch.cuda.max_memory_allocated(device) / (1024**3)
    model_size_gb = sum(p.numel() for p in model.parameters()) * 2 / (1024**3)  # fp16
    results["Model_Size(GB)"] = round(model_size_gb, 2)
    results["Peak_GPU_Memory(GB)"] = round(gpu_mem, 2)

    # Save CSV locally
    df_model = pd.DataFrame([results])
    file_name = f"{model_name.replace('/', '_')}_results.csv"
    df_model.to_csv(file_name, index=False)
    print(f"✅ Saved results for {model_name} to {file_name}")

    del model, tokenized, output
    torch.cuda.empty_cache()
    gc.collect()

    return results

# ============================================================
# 🧩 STEP 6: Run models
# ============================================================
models = [
    "Qwen/Qwen3-4B-Instruct-2507",
    "Qwen/Qwen3-1.7B"
]

results_all = {}
for m in models:
    results_all[m] = run_inference_batched(m, test_data, num_samples=100, batch_size=8)

# ============================================================
# 🧩 STEP 7: Display results summary
# ============================================================
df = pd.DataFrame(results_all).T
print("\n📊 Baseline Vanilla Inference Results (100 GSM8K samples):")
display(df)


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 11.3 MB/s eta 0:00:00
Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Generating answers (batched): 100%|██████████| 13/13 [00:45<00:00,  3.52s/it]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Saved results for Qwen/Qwen3-1.7B to Qwen_Qwen3-1.7B_results.csv

📊 Baseline Vanilla Inference Results (100 GSM8K samples):


,ROUGE_rouge1,ROUGE_rouge2,ROUGE_rougeL,ROUGE_rougeLsum,BERTScore_F1,Exact_Match,Accuracy,Inference_Time(s),Model_Size(GB),Peak_GPU_Memory(GB)
Qwen/Qwen3-4B-Instruct-2507,0.038828,0.000933,0.038874,0.039003,0.777004,0.03,0.03,61.23,7.49,9.11
Qwen/Qwen3-1.7B,0.038553,0.000933,0.038613,0.038737,0.777463,0.02,0.02,45.74,3.20,9.11
